In [0]:
from pyspark.sql import functions as F

def normalize_columns(df):
    """Standardizes column names: lowercase, spaces to underscores."""
    for c in df.columns:
        df = df.withColumnRenamed(c, c.strip().lower().replace(" ", "_"))
    return df

In [0]:
df_budget = spark.table("workspace.bronze.ns_budget")
df_budget = normalize_columns(df_budget)

df_budget = (
    df_budget
    .drop("_file", "_line", "_modified", "_fivetran_synced")
    .withColumn("ns_store_id", F.col("ns_store_id").cast("int"))
    .withColumn("budget_amount", F.col("budget_amount").cast("double"))
    .dropDuplicates()
)

df_budget.write.mode("overwrite").option("overwriteSchema", "true").format("delta").saveAsTable("workspace.silver.ns_budget")
print(f"silver.ns_budget: {df_budget.count()} rows written")